## CTG Classification using Pre-Trained ResNet-50 Model

This notebook trains a ResNet-50 model on the CWT spectrograms generated from fetal heart rate signals.
The spectrograms are pre-organized into train/validation/test sets and class folders (normal/distressed).

In [9]:
# Import necessary libraries
import os
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    auc
)

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [10]:
class paths:
    output_folder = 'outputs/spectograms'
    train = os.path.join(output_folder, 'train')
    validation = os.path.join(output_folder, 'validation')
    test = os.path.join(output_folder, 'test')

IMG_SIZE = (224, 224)
BATCH_SIZE = 32


In [ ]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    paths.train,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=True,
    seed=42
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    paths.validation,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=False
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    paths.test,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=False
)


NameError: name 'image_dataset_from_directory' is not defined

In [3]:
''' 
Build ResNet-50 Model 
'''

def build_resnet50_model(input_shape=(224, 224, 3)):
    base_model = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=input_shape
    )

    # Freeze backbone (CRUCIAL for your dataset size)
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation="sigmoid")(x)

    model = Model(inputs=base_model.input, outputs=output)
    return model



In [4]:
loss_fn = tf.keras.losses.BinaryFocalCrossentropy(
    gamma=2.0,
    alpha=0.25
)


In [5]:
# Compile the model
model = build_resnet50_model()

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=loss_fn,
    metrics=[
        tf.keras.metrics.AUC(curve="PR", name="pr_auc"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.Precision(name="precision")
    ]
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,850,113 (90.98 MB)

 Trainable params: 262,401 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [6]:
# Callbacks
callbacks = [
    EarlyStopping(
        monitor="val_pr_auc",
        patience=3,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_pr_auc",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]



In [7]:
# Train 
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    callbacks=callbacks
)




NameError: name 'train_dataset' is not defined

In [ ]:
y_true = []
y_scores = []

for x_batch, y_batch in test_dataset:
    preds = model.predict(x_batch).flatten()
    y_scores.extend(preds)
    y_true.extend(y_batch.numpy())

y_true = np.array(y_true)
y_scores = np.array(y_scores)


In [ ]:
print("Pred prob range:", y_scores.min(), y_scores.max())
plt.hist(y_scores, bins=30)
plt.title("Predicted probability distribution")
plt.show()


In [ ]:
precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
pr_auc = auc(recall, precision)

print("PR-AUC:", pr_auc)


In [ ]:
for t in np.arange(0.1, 0.61, 0.05):
    y_pred = (y_scores >= t).astype(int)

    p = tf.keras.metrics.Precision()
    r = tf.keras.metrics.Recall()

    p.update_state(y_true, y_pred)
    r.update_state(y_true, y_pred)

    print(
        f"Threshold {t:.2f} | "
        f"Precision {p.result().numpy():.3f} | "
        f"Recall {r.result().numpy():.3f}"
    )


In [ ]:
idx = np.where(recall >= 0.7)[0]

best_idx = idx[-1]
best_threshold = thresholds[best_idx]

print("Chosen threshold:", best_threshold)
print("Precision:", precision[best_idx])
print("Recall:", recall[best_idx])


In [ ]:
y_pred_final = (y_scores >= best_threshold).astype(int)

cm = confusion_matrix(y_true, y_pred_final)
print("Confusion Matrix:\n", cm)

print(
    classification_report(
        y_true,
        y_pred_final,
        digits=4,
        zero_division=0
    )
)
